# M5 Forecasting Accuracy - Colab Pipeline

ローカル環境のメモリ制約を回避するため、Colab上で全工程を実行する。

## 構成
1. Google Driveマウント
2. 生データ読み込み・melt
3. 特徴量生成（calendar / price / lag / rolling / encoding）
4. LightGBM単一モデル訓練（直近データ + 28日ホールドアウト検証）
5. d_1942〜d_1969予測 → submission.csv生成 → Driveへ保存

## 設計上のポイント
- **再帰予測不要**: 予測期間28日とlag_28が一致するため直接予測可能
- **Tweedie目的関数**: M5は売上0が多い・heavy-tailedなのでTweedieが定番
- **Drive中継**: 入力CSV5ファイルとsubmission.csvをDrive経由で授受

## 0. ライブラリ準備

In [ ]:
import gc
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error

print(f'pandas: {pd.__version__}')
print(f'lightgbm: {lgb.__version__}')

T0 = time.time()

## 1. Google Driveマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_DIR = Path('/content/drive/MyDrive/Colab Data/m5-forecasting-accuracy')
INPUT_DIR = DRIVE_DIR / 'input'
OUTPUT_DIR = DRIVE_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for f in ['calendar.csv', 'sell_prices.csv', 'sales_train_evaluation.csv', 'sample_submission.csv']:
    p = INPUT_DIR / f
    print(f'{f}: {"OK" if p.exists() else "MISSING"} ({p.stat().st_size/1024**2:.1f} MB)' if p.exists() else f'{f}: MISSING')

## 2. データ読み込み・melt

In [ ]:
# Calendar
calendar = pd.read_csv(INPUT_DIR / 'calendar.csv', parse_dates=['date'])
calendar['d_num'] = calendar['d'].str[2:].astype(int)
print(f'calendar: {calendar.shape}, d_num範囲={calendar["d_num"].min()}〜{calendar["d_num"].max()}')

# Sales（dtype指定で省メモリ）
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
d_cols = [f'd_{i}' for i in range(1, 1942)]
dtype_dict = {c: 'int16' for c in d_cols}
dtype_dict.update({c: 'category' for c in id_cols})
sales = pd.read_csv(INPUT_DIR / 'sales_train_evaluation.csv', dtype=dtype_dict)
print(f'sales: {sales.shape}')

# evaluation期間（d_1942〜d_1969）の予測用プレースホルダ列を追加
for d in range(1942, 1970):
    sales[f'd_{d}'] = np.int16(0)
print(f'sales (with eval placeholder): {sales.shape}')

In [ ]:
# Wide → Long
all_d_cols = [f'd_{i}' for i in range(1, 1970)]
df = sales.melt(id_vars=id_cols, value_vars=all_d_cols, var_name='d', value_name='sales')
df['d_num'] = df['d'].str[2:].astype('int16')
df.drop(columns=['d'], inplace=True)
df['sales'] = df['sales'].astype('int16')
print(f'long format: {df.shape}, メモリ={df.memory_usage(deep=True).sum()/1024**2:.0f} MB')

del sales
gc.collect()

In [ ]:
# Calendarをマージ
cal_keep = ['d_num', 'date', 'wm_yr_wk', 'wday', 'month', 'year',
            'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']
df = df.merge(calendar[cal_keep], on='d_num', how='left')

# dtype最適化
df['wday'] = df['wday'].astype('int8')
df['month'] = df['month'].astype('int8')
df['year'] = df['year'].astype('int16')
df['snap_CA'] = df['snap_CA'].astype('int8')
df['snap_TX'] = df['snap_TX'].astype('int8')
df['snap_WI'] = df['snap_WI'].astype('int8')
df['event_name_1'] = df['event_name_1'].astype('category')
df['event_type_1'] = df['event_type_1'].astype('category')

print(f'merged: {df.shape}, メモリ={df.memory_usage(deep=True).sum()/1024**2:.0f} MB')

## 3. 直近データへ絞る（メモリ対策）

lag_56＋ローリング60日分の助走と十分な学習データを確保するため、`d_1500` 以降のみ保持する。

In [ ]:
df = df[df['d_num'] >= 1500].reset_index(drop=True)
gc.collect()
print(f'filtered: {df.shape}, メモリ={df.memory_usage(deep=True).sum()/1024**2:.0f} MB')

## 4. 特徴量生成

In [ ]:
# 4-1. カレンダー特徴量
df['day_of_month'] = df['date'].dt.day.astype('int8')
df['week_of_year'] = df['date'].dt.isocalendar().week.astype('int8')
df['is_weekend'] = df['wday'].isin([1, 2]).astype('int8')
df['has_event'] = df['event_name_1'].notna().astype('int8')

# 州ごとのSNAPフラグを1列に集約
df['snap'] = 0
for state in ['CA', 'TX', 'WI']:
    mask = df['state_id'] == state
    df.loc[mask, 'snap'] = df.loc[mask, f'snap_{state}']
df['snap'] = df['snap'].astype('int8')

# 不要列削除
df.drop(columns=['snap_CA', 'snap_TX', 'snap_WI', 'event_name_1', 'event_type_1', 'date'], inplace=True)
gc.collect()
print(f'after calendar: {df.shape}, メモリ={df.memory_usage(deep=True).sum()/1024**2:.0f} MB')

In [ ]:
# 4-2. 価格特徴量
sell_prices = pd.read_csv(INPUT_DIR / 'sell_prices.csv')
df = df.merge(
    sell_prices[['store_id', 'item_id', 'wm_yr_wk', 'sell_price']],
    on=['store_id', 'item_id', 'wm_yr_wk'], how='left'
)
df['sell_price'] = df['sell_price'].astype('float32')

# 価格派生
price_stats = sell_prices.groupby(['store_id', 'item_id'])['sell_price'].agg(
    price_max='max', price_min='min', price_mean='mean'
).reset_index()
df = df.merge(price_stats, on=['store_id', 'item_id'], how='left')
df['price_discount'] = ((df['price_max'] - df['sell_price']) / df['price_max']).astype('float32')
df['price_norm'] = ((df['sell_price'] - df['price_mean']) / df['price_mean']).astype('float32')
for c in ['price_max', 'price_min', 'price_mean']:
    df[c] = df[c].astype('float32')

del sell_prices, price_stats
gc.collect()
print(f'after price: {df.shape}, メモリ={df.memory_usage(deep=True).sum()/1024**2:.0f} MB')

In [ ]:
# 4-3. ラグ特徴量（28日以降のみ。予測時にも使えるラグに限定）
df = df.sort_values(['id', 'd_num']).reset_index(drop=True)
for lag in [28, 35, 42, 49, 56]:
    df[f'lag_{lag}'] = df.groupby('id', observed=True)['sales'].shift(lag).astype('float32')
    print(f'  lag_{lag} done')
gc.collect()

In [ ]:
# 4-4. ローリング特徴量（lag_base=28でshiftしてからrolling）
shifted = df.groupby('id', observed=True)['sales'].shift(28)
for w in [7, 14, 28, 60]:
    df[f'rolling_mean_{w}'] = (
        shifted.groupby(df['id'], observed=True)
        .transform(lambda x: x.rolling(w, min_periods=1).mean())
        .astype('float32')
    )
    print(f'  rolling_mean_{w} done')

# 標準偏差は短期のみ
for w in [7, 28]:
    df[f'rolling_std_{w}'] = (
        shifted.groupby(df['id'], observed=True)
        .transform(lambda x: x.rolling(w, min_periods=1).std())
        .astype('float32')
    )
    print(f'  rolling_std_{w} done')

del shifted
gc.collect()

In [ ]:
# 4-5. カテゴリエンコーディング
for col in ['store_id', 'item_id', 'dept_id', 'cat_id', 'state_id']:
    df[f'{col}_enc'] = df[col].astype(str).factorize()[0].astype('int16')

print(f'final df: {df.shape}, メモリ={df.memory_usage(deep=True).sum()/1024**2:.0f} MB')
print(f'特徴量生成までの経過: {(time.time()-T0)/60:.1f} 分')

## 5. 訓練・検証分割

- 学習期間: `d_1556` ～ `d_1885`（lag_56で先頭56日落ちる）
- 検証期間: `d_1886` ～ `d_1913`（直近28日。Public LBに最も近い）
- 予測対象: `d_1942` ～ `d_1969`（evaluation。`d_1914`～`d_1941`はvalidation実測値で埋める）

In [ ]:
feature_cols = [
    'wday', 'month', 'year', 'day_of_month', 'week_of_year',
    'is_weekend', 'has_event', 'snap',
    'sell_price', 'price_max', 'price_min', 'price_mean', 'price_discount', 'price_norm',
    'lag_28', 'lag_35', 'lag_42', 'lag_49', 'lag_56',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_mean_60',
    'rolling_std_7', 'rolling_std_28',
    'store_id_enc', 'item_id_enc', 'dept_id_enc', 'cat_id_enc', 'state_id_enc',
]
print(f'特徴量数: {len(feature_cols)}')

# lag_56が有効になる範囲のみ学習対象
train_mask = (df['d_num'] >= 1556) & (df['d_num'] <= 1913) & df['lag_56'].notna()
val_mask = (df['d_num'] >= 1886) & (df['d_num'] <= 1913) & df['lag_56'].notna()
trn_only_mask = train_mask & ~val_mask

X_train = df.loc[trn_only_mask, feature_cols]
y_train = df.loc[trn_only_mask, 'sales'].astype('float32')
X_valid = df.loc[val_mask, feature_cols]
y_valid = df.loc[val_mask, 'sales'].astype('float32')

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')

## 6. LightGBM訓練

In [ ]:
params = {
    'objective': 'tweedie',
    'tweedie_variance_power': 1.1,
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 128,
    'min_data_in_leaf': 100,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l2': 0.1,
    'verbose': -1,
    'n_jobs': -1,
    'force_row_wise': True,
}

train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

model = lgb.train(
    params,
    train_data,
    num_boost_round=2000,
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

print(f'best_iteration: {model.best_iteration}')
print(f'best valid RMSE: {model.best_score["valid"]["rmse"]:.4f}')
print(f'訓練までの経過: {(time.time()-T0)/60:.1f} 分')

In [ ]:
# 特徴量重要度
imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)
print(imp.head(20).to_string(index=False))

## 7. 予測とsubmission生成

In [ ]:
# evaluation期間（d_1942〜d_1969）の予測
eval_mask = (df['d_num'] >= 1942) & (df['d_num'] <= 1969)
test_df = df.loc[eval_mask, ['id', 'd_num'] + feature_cols].copy()
test_df['pred'] = model.predict(test_df[feature_cols], num_iteration=model.best_iteration)
test_df['pred'] = test_df['pred'].clip(lower=0)
print(f'eval予測数: {len(test_df)}, mean={test_df["pred"].mean():.3f}')

# validation期間（d_1914〜d_1941）の実測値
val_mask_full = (df['d_num'] >= 1914) & (df['d_num'] <= 1941)
val_actual = df.loc[val_mask_full, ['id', 'd_num', 'sales']].copy()
val_actual['sales'] = val_actual['sales'].clip(lower=0).astype('float32')
print(f'val実測数: {len(val_actual)}, mean={val_actual["sales"].mean():.3f}')

In [ ]:
# F列を割り当て
test_df['F'] = 'F' + (test_df['d_num'] - 1941).astype(int).astype(str)
val_actual['F'] = 'F' + (val_actual['d_num'] - 1913).astype(int).astype(str)

# pivot
eval_pivot = test_df.pivot(index='id', columns='F', values='pred').reset_index()
val_pivot = val_actual.pivot(index='id', columns='F', values='sales').reset_index()

# id末尾を _validation / _evaluation に整形
# sales_train_evaluationのidは '_evaluation' で終わるが念のため確認
print(f'sample id: {eval_pivot["id"].iloc[0]}')

# evaluationは _evaluation 付き、validationは _validation 付き
eval_pivot['id'] = eval_pivot['id'].astype(str).str.replace('_evaluation', '') + '_evaluation'
val_pivot['id'] = val_pivot['id'].astype(str).str.replace('_evaluation', '') + '_validation'

submission = pd.concat([val_pivot, eval_pivot], ignore_index=True)
f_cols = [f'F{i}' for i in range(1, 29)]
submission = submission[['id'] + f_cols]
print(f'submission: {submission.shape}')
print(submission.head())

In [ ]:
# sample_submissionと整合性確認
sample_sub = pd.read_csv(INPUT_DIR / 'sample_submission.csv')
print(f'sample: {sample_sub.shape}, ours: {submission.shape}')

# id順を sample に合わせる
submission = sample_sub[['id']].merge(submission, on='id', how='left')
print(f'merged: {submission.shape}')
print(f'NaN残: {submission[f_cols].isna().sum().sum()}')

# NaNがあれば0埋め
submission[f_cols] = submission[f_cols].fillna(0)

# 保存
out_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(out_path, index=False)
print(f'saved: {out_path} ({out_path.stat().st_size/1024**2:.1f} MB)')
print(f'全体経過時間: {(time.time()-T0)/60:.1f} 分')

## 完了

Driveの `output/submission.csv` に保存された。これをローカルに取得して `kaggle competitions submit` で提出する。